# LineupAI

# 01 — Dataset + Modelado

- **Fuentes:** API Fantasy Premier League (bootstrap-static, element-summary)
- **Objetivo:** Predecir puntos del siguiente GW por jugador

## Preparación de ambiente

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import json
from pathlib import Path
from typing import Dict, Any, List, Optional
import requests
from tqdm.auto import tqdm
import joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

ARTIFACTS_DIR = Path("artifacts")

pd.set_option("display.max_columns", 200)

## 1) Data Wrangling


In [4]:
FPL_BOOTSTRAP = "https://fantasy.premierleague.com/api/bootstrap-static/"
FPL_PLAYER_SUMMARY = "https://fantasy.premierleague.com/api/element-summary/{player_id}/"

CACHE_DIR = Path("cache_fpl")
CACHE_DIR.mkdir(exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "LineupAI-DatasetBuilder/1.0"})

def _cache_path(key: str) -> Path:
    safe = key.replace("https://", "").replace("/", "_").replace("?", "_")
    return CACHE_DIR / f"{safe}.json"

def get_json(url: str, ttl_seconds: int = 3600) -> Dict[str, Any]:
    path = _cache_path(url)
    if path.exists():
        age = time.time() - path.stat().st_mtime
        if age <= ttl_seconds:
            return json.loads(path.read_text(encoding="utf-8"))

    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()
    path.write_text(json.dumps(data), encoding="utf-8")
    return data

bootstrap = get_json(FPL_BOOTSTRAP, ttl_seconds=3600)
print("bootstrap keys:", list(bootstrap.keys())[:10])


bootstrap keys: ['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements']


In [5]:
players = pd.DataFrame(bootstrap["elements"])
teams = pd.DataFrame(bootstrap["teams"])[["id", "name", "short_name"]].rename(columns={"id":"team_id"})
pos = pd.DataFrame(bootstrap["element_types"])[["id", "singular_name_short"]].rename(columns={"id":"pos_id"})
events = pd.DataFrame(bootstrap["events"])[["id","is_current","is_next","finished","deadline_time"]].rename(columns={"id":"gw"})

players = players.merge(teams, left_on="team", right_on="team_id", how="left")
players = players.merge(pos, left_on="element_type", right_on="pos_id", how="left")

players["now_cost_m"] = players["now_cost"] / 10.0
players.rename(columns={"singular_name_short":"position"}, inplace=True)

players[["id","web_name","short_name","position","now_cost_m","status","chance_of_playing_next_round","minutes","total_points","goals_scored","assists","clean_sheets","saves"]].head()


,id,web_name,short_name,position,now_cost_m,status,chance_of_playing_next_round,minutes,total_points,goals_scored,assists,clean_sheets,saves
0,1,Raya,ARS,GKP,6.0,a,NaN,2340,108,0,0,13,37
1,2,Arrizabalaga,ARS,GKP,4.1,a,NaN,0,0,0,0,0,0
2,3,Hein,ARS,GKP,4.0,u,0.0,0,0,0,0,0,0
3,4,Setford,ARS,GKP,3.9,a,NaN,0,0,0,0,0,0
4,5,Gabriel,ARS,DEF,7.1,a,100.0,1715,138,3,2,12,0


## 2) Selección de jugadores elegibles


In [6]:
candidate_players = players.loc[(players["minutes"] > 0) | (players["status"] != "u")].copy()
candidate_ids = candidate_players["id"].astype(int).tolist()
len(candidate_ids), candidate_players[["web_name","short_name","position","minutes","status"]].head()


(643,
        web_name short_name position  minutes status
 0          Raya        ARS      GKP     2340      a
 1  Arrizabalaga        ARS      GKP        0      a
 3       Setford        ARS      GKP        0      a
 4       Gabriel        ARS      DEF     1715      a
 5        Saliba        ARS      DEF     1714      a)

## 3) Descargar history por jugador (element-summary)


In [7]:
def fetch_player_history(player_id: int, ttl_seconds: int = 3600) -> pd.DataFrame:
    data = get_json(FPL_PLAYER_SUMMARY.format(player_id=player_id), ttl_seconds=ttl_seconds)
    hist = pd.DataFrame(data.get("history", []))
    if hist.empty:
        return hist
    hist["player_id"] = int(player_id)
    hist.rename(columns={"round":"gw"}, inplace=True)
    return hist

all_hist = []
for pid in tqdm(candidate_ids, desc="Fetching element-summary histories"):
    try:
        h = fetch_player_history(pid, ttl_seconds=3600)
        if not h.empty:
            all_hist.append(h)
    except Exception:
        continue

gw_hist = pd.concat(all_hist, ignore_index=True) if all_hist else pd.DataFrame()
gw_hist.shape, gw_hist.head()


Fetching element-summary histories:   0%|          | 0/643 [00:00<?, ?it/s]

((15428, 42),
    element  fixture  opponent_team  total_points  was_home  \
 0        1        9             14            10     False   
 1        1       11             11             6      True   
 2        1       25             12             2     False   
 3        1       31             16             6      True   
 4        1       41             13             2      True   
 
            kickoff_time  team_h_score  team_a_score  gw  modified  minutes  \
 0  2025-08-17T15:30:00Z           0.0           1.0   1     False       90   
 1  2025-08-23T16:30:00Z           5.0           0.0   2     False       90   
 2  2025-08-31T15:30:00Z           1.0           0.0   3     False       90   
 3  2025-09-13T11:30:00Z           3.0           0.0   4     False       90   
 4  2025-09-21T15:30:00Z           1.0           1.0   5     False       90   
 
    goals_scored  assists  clean_sheets  goals_conceded  own_goals  \
 0             0        0             1               0     

## 4) Merge + limpieza base

In [8]:
wanted = [
    "player_id","gw",
    "minutes","total_points",
    "goals_scored","assists","clean_sheets","goals_conceded","saves","bps",
    "expected_goals","expected_assists","expected_goal_involvements","expected_goals_conceded",
]
cols = [c for c in wanted if c in gw_hist.columns]
df = gw_hist[cols].copy()

meta = players.rename(columns={"id":"player_id"})[
    ["player_id","web_name","short_name","team_id","position","now_cost_m","status","chance_of_playing_next_round"]
].copy()

df = df.merge(meta, on="player_id", how="left")

num_cols = [c for c in df.columns if c not in {"web_name","short_name","position","status"}]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.sort_values(["player_id","gw"], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()


,player_id,gw,minutes,total_points,goals_scored,assists,clean_sheets,goals_conceded,saves,bps,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,web_name,short_name,team_id,position,now_cost_m,status,chance_of_playing_next_round
0,1,1,90,10,0,0,1,0,7,38,0.0,0.00,0.00,1.52,Raya,ARS,1,GKP,6.0,a,NaN
1,1,2,90,6,0,0,1,0,1,28,0.0,0.00,0.00,0.17,Raya,ARS,1,GKP,6.0,a,NaN
2,1,3,90,2,0,0,0,1,2,12,0.0,0.02,0.02,0.52,Raya,ARS,1,GKP,6.0,a,NaN
3,1,4,90,6,0,0,1,0,1,24,0.0,0.00,0.00,0.20,Raya,ARS,1,GKP,6.0,a,NaN
4,1,5,90,2,0,0,0,1,2,13,0.0,0.01,0.01,0.89,Raya,ARS,1,GKP,6.0,a,NaN


## 5) Feature engineering

In [9]:
WINDOW = 5

def add_rolling_features(df_in: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    df = df_in.copy().sort_values(["player_id","gw"])
    base_feats = [
        "total_points","minutes",
        "goals_scored","assists",
        "clean_sheets","goals_conceded","saves","bps",
        "expected_goals","expected_assists",
    ]
    base_feats = [c for c in base_feats if c in df.columns]

    g = df.groupby("player_id", group_keys=False)

    for c in base_feats:
        df[f"{c}_roll_{window}"] = g[c].apply(lambda s: s.shift(1).rolling(window, min_periods=1).sum())

    # derived xGI roll if expected available
    eg = f"expected_goals_roll_{window}"
    ea = f"expected_assists_roll_{window}"
    if eg in df.columns and ea in df.columns:
        df[f"xGI_roll_{window}"] = df[eg] + df[ea]

    for c in ["total_points","minutes","bps"]:
        df[f"{c}_mean_{window}"] = g[c].apply(lambda s: s.shift(1).rolling(window, min_periods=1).mean())

    return df

df_feat = add_rolling_features(df, window=WINDOW)
df_feat[["player_id","gw","total_points",f"total_points_roll_{WINDOW}"]].head(10)


,player_id,gw,total_points,total_points_roll_5,target_total_points_next_gw
0,1,1,10,NaN,6.0
1,1,2,6,10.0,2.0
2,1,3,2,16.0,6.0
3,1,4,6,18.0,2.0
4,1,5,2,24.0,2.0
5,1,6,2,26.0,6.0
6,1,7,6,18.0,6.0
7,1,8,6,18.0,6.0
8,1,9,6,22.0,6.0
9,1,10,6,22.0,1.0


## 6) Target: y_points_next_gw

In [ ]:
df_feat["y_points_next_gw"] = df_feat.groupby("player_id")["total_points"].shift(-1)
df_feat["minutes_next_gw"] = df_feat.groupby("player_id")["minutes"].shift(-1)
df_feat["y_play_next_gw"] = (df_feat["minutes_next_gw"] > 0).astype(int)

data_v2 = df_feat.dropna(subset=["y_points_next_gw", "minutes_next_gw"]).copy()
data_v2[["y_points_next_gw", "y_play_next_gw"]].describe()

## 7) Walk-forward CV

In [ ]:
MIN_TRAIN_GW = 10
VAL_BLOCK = 3

def make_folds(gw_min: int, gw_max: int) -> list:
    folds = []
    train_end = max(gw_min + MIN_TRAIN_GW - 1, gw_min)
    while True:
        val_start = train_end + 1
        val_end = val_start + VAL_BLOCK - 1
        if val_end > gw_max:
            break
        folds.append((train_end, val_start, val_end))
        train_end = val_end
    return folds

folds = make_folds(int(data_v2["gw"].min()), int(data_v2["gw"].max()))
print("Folds (train_end, val_start, val_end):", folds)

---
## RUTA A — Modelo IA (baseline automático)

### 8A) Baseline "auto-features" + walk-forward

In [ ]:

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

target_col = "y_points_next_gw"
roll_cols = [c for c in data_v2.columns if c.endswith(f"_roll_{WINDOW}")]
numeric_roll_cols = [c for c in roll_cols if pd.api.types.is_numeric_dtype(data_v2[c])]
feature_cols_auto = numeric_roll_cols + ["now_cost_m"]
if "chance_of_playing_next_round" in data_v2.columns:
    feature_cols_auto.append("chance_of_playing_next_round")
feature_cols_auto = [c for c in feature_cols_auto if c in data_v2.columns]

X_all = data_v2[feature_cols_auto].fillna(0)
y_all = data_v2[target_col].astype(float)
gw_all = data_v2["gw"].astype(int)

results = []
for i, (tr_end, v_start, v_end) in enumerate(folds, start=1):
    tr_idx = gw_all <= tr_end
    va_idx = (gw_all >= v_start) & (gw_all <= v_end)
    X_train, y_train = X_all.loc[tr_idx], y_all.loc[tr_idx]
    X_val, y_val = X_all.loc[va_idx], y_all.loc[va_idx]

    ridge = Ridge(alpha=1.0, random_state=42)
    ridge.fit(X_train, y_train)
    hgb = HistGradientBoostingRegressor(random_state=42, max_depth=3, learning_rate=0.05)
    hgb.fit(X_train, y_train)

    results.append({
        "fold": i, "train_gw_end": tr_end, "val_gw_start": v_start, "val_gw_end": v_end,
        "ridge_mae": mean_absolute_error(y_val, ridge.predict(X_val)),
        "ridge_rmse": rmse(y_val, ridge.predict(X_val)),
        "hgb_mae": mean_absolute_error(y_val, hgb.predict(X_val)),
        "hgb_rmse": rmse(y_val, hgb.predict(X_val)),
    })

res_df = pd.DataFrame(results)
display(res_df)
print("\nPromedios:")
print("Ridge MAE:", round(res_df["ridge_mae"].mean(), 4), "RMSE:", round(res_df["ridge_rmse"].mean(), 4))
print("HGB   MAE:", round(res_df["hgb_mae"].mean(), 4), "RMSE:", round(res_df["hgb_rmse"].mean(), 4))


Train: (13439, 15) Val: (1346, 15) Val GWs: [24, 25, 26]


---
## RUTA B — Modelo Manual (final)

### 8B) Features


In [ ]:
ROLL_COLS = [
    "total_points_roll_5", "minutes_roll_5", "goals_scored_roll_5", "assists_roll_5",
    "clean_sheets_roll_5", "goals_conceded_roll_5", "saves_roll_5", "bps_roll_5",
    "expected_goals_roll_5", "expected_assists_roll_5", "xGI_roll_5",
]
feature_cols = ROLL_COLS + ["now_cost_m", "chance_of_playing_next_round"]
feature_cols = [c for c in feature_cols if c in data_v2.columns]
data_v2["chance_of_playing_next_round"] = pd.to_numeric(data_v2["chance_of_playing_next_round"], errors="coerce")
print("Features manuales (13):", feature_cols)

### 9B) Walk-forward manual (Ridge vs HGB)

In [ ]:


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

WINDOW = 5
VAL_BLOCK = 3
MIN_TRAIN_GW = 10
target_col = "y_points_next_gw"

# Asegurar que chance sea numérica
data_v2 = data_v2.copy()
data_v2["chance_of_playing_next_round"] = pd.to_numeric(
    data_v2["chance_of_playing_next_round"], errors="coerce"
)

feature_cols = [
    "total_points_roll_5", "minutes_roll_5", "goals_scored_roll_5", "assists_roll_5",
    "clean_sheets_roll_5", "goals_conceded_roll_5", "saves_roll_5", "bps_roll_5",
    "expected_goals_roll_5", "expected_assists_roll_5", "xGI_roll_5",
    "now_cost_m", "chance_of_playing_next_round"
]
feature_cols = [c for c in feature_cols if c in data_v2.columns]

print("Using features:", feature_cols)
print("n_features:", len(feature_cols))

X_all = data_v2[feature_cols].fillna(0)
y_all = data_v2[target_col].astype(float)
gw_all = data_v2["gw"].astype(int)

max_gw = int(gw_all.max())
min_gw = int(gw_all.min())

folds = []
train_end = max(min_gw + MIN_TRAIN_GW - 1, min_gw)
while True:
    val_start = train_end + 1
    val_end = val_start + VAL_BLOCK - 1
    if val_end > max_gw:
        break
    folds.append((train_end, val_start, val_end))
    train_end = val_end

print("Folds:", folds)

rows = []
for i, (tr_end, v_start, v_end) in enumerate(folds, start=1):
    tr_idx = gw_all <= tr_end
    va_idx = (gw_all >= v_start) & (gw_all <= v_end)

    X_train, y_train = X_all.loc[tr_idx], y_all.loc[tr_idx]
    X_val, y_val = X_all.loc[va_idx], y_all.loc[va_idx]

    ridge = Ridge(alpha=1.0, random_state=42)
    ridge.fit(X_train, y_train)
    pr = ridge.predict(X_val)

    hgb = HistGradientBoostingRegressor(random_state=42, max_depth=3, learning_rate=0.05)
    hgb.fit(X_train, y_train)
    ph = hgb.predict(X_val)

    rows.append({
        "fold": i,
        "train_gw_end": tr_end,
        "val_gw_start": v_start,
        "val_gw_end": v_end,
        "ridge_mae": mean_absolute_error(y_val, pr),
        "ridge_rmse": rmse(y_val, pr),
        "hgb_mae": mean_absolute_error(y_val, ph),
        "hgb_rmse": rmse(y_val, ph),
    })

res_df = pd.DataFrame(rows)
display(res_df)

print("\nPromedios:")
print("Ridge MAE:", round(res_df["ridge_mae"].mean(), 4), "RMSE:", round(res_df["ridge_rmse"].mean(), 4))
print("HGB   MAE:", round(res_df["hgb_mae"].mean(), 4), "RMSE:", round(res_df["hgb_rmse"].mean(), 4))


Using features: ['total_points_roll_5', 'minutes_roll_5', 'goals_scored_roll_5', 'assists_roll_5', 'clean_sheets_roll_5', 'goals_conceded_roll_5', 'saves_roll_5', 'bps_roll_5', 'expected_goals_roll_5', 'expected_assists_roll_5', 'xGI_roll_5', 'now_cost_m', 'chance_of_playing_next_round']
n_features: 13
Folds: [(10, 11, 13), (13, 14, 16), (16, 17, 19), (19, 20, 22), (22, 23, 25)]


/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/javierlozano/Documents/ENTORNOS/CdD/M5/.venv/lib/python3.10/

,fold,train_gw_end,val_gw_start,val_gw_end,ridge_mae,ridge_rmse,hgb_mae,hgb_rmse
0,1,10,11,13,1.582437,2.463070,1.552078,2.462224
1,2,13,14,16,1.520049,2.353815,1.498560,2.366938
2,3,16,17,19,1.548949,2.394054,1.514511,2.379658
3,4,19,20,22,1.468690,2.188791,1.432362,2.170835
4,5,22,23,25,1.400931,2.213186,1.392304,2.228274



Promedios:
Ridge MAE: 1.5042 RMSE: 2.3226
HGB   MAE: 1.478 RMSE: 2.3216


### 10B) Entrenar final y guardar artefactos

In [ ]:


ARTIFACTS_DIR.mkdir(exist_ok=True)

feature_cols_final = [
    "total_points_roll_5", "minutes_roll_5", "goals_scored_roll_5", "assists_roll_5",
    "clean_sheets_roll_5", "goals_conceded_roll_5", "saves_roll_5", "bps_roll_5",
    "expected_goals_roll_5", "expected_assists_roll_5", "xGI_roll_5",
    "now_cost_m", "chance_of_playing_next_round"
]

X = data_v2[feature_cols_final].copy()
X["chance_of_playing_next_round"] = pd.to_numeric(X["chance_of_playing_next_round"], errors="coerce")
X = X.fillna(0)

y = data_v2["y_points_next_gw"].astype(float)

model = HistGradientBoostingRegressor(random_state=42, max_depth=3, learning_rate=0.05)
model.fit(X, y)

joblib.dump(model, ARTIFACTS_DIR / "hgb_total_points_next_gw.joblib")

meta = {
    "model_type": "HistGradientBoostingRegressor",
    "params": {"max_depth": 3, "learning_rate": 0.05, "random_state": 42},
    "window": 5,
    "feature_cols": feature_cols_final,
    "target": "y_points_next_gw",
    "n_rows": int(len(data_v2)),
    "players": int(data_v2["player_id"].nunique()),
    "gw_min": int(data_v2["gw"].min()),
    "gw_max": int(data_v2["gw"].max()),
    "cv": {"type": "walk_forward", "val_block": 3, "min_train_gw": 10},
    "metrics_cv_mean": {"mae": 1.4780, "rmse": 2.3216},
}
(ARTIFACTS_DIR / "hgb_total_points_next_gw_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Saved:", (ARTIFACTS_DIR / "hgb_total_points_next_gw.joblib").resolve())
print("Saved:", (ARTIFACTS_DIR / "hgb_total_points_next_gw_meta.json").resolve())


Saved: /Users/javierlozano/Documents/ENTORNOS/CdD/M5/artifacts/hgb_total_points_next_gw.joblib
Saved: /Users/javierlozano/Documents/ENTORNOS/CdD/M5/artifacts/hgb_total_points_next_gw_meta.json


## 11) Export dataset


In [16]:
ARTIFACTS_DIR.mkdir(exist_ok=True)
data_path = ARTIFACTS_DIR / "fpl_player_gw_dataset.parquet"
data_v2.to_parquet(data_path, index=False)
print("Saved:", data_path.resolve())


Saved: /Users/javierlozano/Documents/ENTORNOS/CdD/M5/artifacts/fpl_player_gw_dataset.parquet
